# Kode dibawah adalah hasil experiment yang dilakukan setelah training

In [2]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/Segmentation_Assignment3_Viskom"
YOLO_ZIP = f"{DRIVE_ROOT}/data/panoptic-indoor-segmentation.v1-train-v1.yolov11.zip"
RUNS_DIR = f"{DRIVE_ROOT}/runs"
!mkdir -p /content/data/yolo
!unzip -q -n "{YOLO_ZIP}" -d /content/data/yolo
!pip -q install ultralytics

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os, glob, shutil, yaml
d = yaml.safe_load(open("/content/data/yolo/data.yaml"))
names = d["names"]
if isinstance(names, dict):
    names = [names[i] for i in sorted(names)]
idx = {n: i for i, n in enumerate(names)}
cmap = {idx["wall"]: idx["wall-other-merged"], idx["paper-merged"]: idx["paper"]}
SRC = "/content/data/yolo/test"
for tag in ["own", "own_h"]:
    os.makedirs(f"/content/data/{tag}/images", exist_ok=True)
    os.makedirs(f"/content/data/{tag}/labels", exist_ok=True)
for img in glob.glob(f"{SRC}/images/*"):
    base = os.path.basename(img)
    if not base.startswith("IMG_20261117_"):
        continue
    stem = os.path.splitext(base)[0]
    lbl = f"{SRC}/labels/{stem}.txt"
    for tag in ["own", "own_h"]:
        shutil.copy(img, f"/content/data/{tag}/images/{base}")
    rows = [l.split() for l in open(lbl)] if os.path.exists(lbl) else []
    with open(f"/content/data/own/labels/{stem}.txt", "w") as f:
        f.write("\n".join(" ".join(p) for p in rows if p) + "\n")
    with open(f"/content/data/own_h/labels/{stem}.txt", "w") as f:
        f.write("\n".join(" ".join([str(cmap.get(int(p[0]), int(p[0])))] + p[1:]) for p in rows if p) + "\n")
for tag in ["own", "own_h"]:
    y = dict(d)
    y.update(path=f"/content/data/{tag}", train="images", val="images", test="images")
    yaml.safe_dump(y, open(f"/content/data/{tag}.yaml", "w"))
    print(tag, len(os.listdir(f"/content/data/{tag}/images")), "images")

own 50 images
own_h 50 images


In [4]:
from ultralytics import YOLO
import pandas as pd
runs = ["yolo11s_default_aug-3", "yolo11s_no_aug", "yolo11s_custom_aug"]
rows = []
for run in runs:
    m = YOLO(f"{RUNS_DIR}/{run}/weights/best.pt")
    for tag in ["own", "own_h"]:
        r = m.val(data=f"/content/data/{tag}.yaml", imgsz=640, batch=16, verbose=False,
                  project=RUNS_DIR, name=f"remap_{run}_{tag}")
        rows.append(dict(run=run, labels=tag, mask_map50=r.seg.map50, mask_map5095=r.seg.map))
df = pd.DataFrame(rows)
print(df)
df.to_csv(f"{RUNS_DIR}/remap_experiment.csv", index=False)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.216 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11s-seg summary (fused): 113 layers, 10,095,454 parameters, 0 gradients, 33.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3455.9±1254.0 MB/s, size: 573.3 KB)
val: Scanning /content/data/own/labels... 50 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 433.7it/s 0.1s
val: New cache created: /content/data/own/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 14.6s/it 58.5s
                   all         50        617      0.397      0.103      0.111     0.0737      0.394     